In [ ]:
from sklearnex import patch_sklearn
patch_sklearn()

import pandas as pd
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer
import time
from sklearn.preprocessing import LabelEncoder

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
test['date'] = pd.to_datetime(test['date'])
train['date'] = pd.to_datetime(train['date'])

In [ ]:
test.head()

In [ ]:
train.head()

In [ ]:
test_with_seasonal = test.copy()
train_with_seasonal = train.copy()

In [ ]:
test_with_seasonal["month"] = test_with_seasonal["date"].dt.month
test_with_seasonal["dayofweek"] = test_with_seasonal["date"].dt.dayofweek
test_with_seasonal["quarter"] = test_with_seasonal["date"].dt.quarter
test_with_seasonal["day"] = test_with_seasonal["date"].dt.day
test_with_seasonal["weekofyear"] = test_with_seasonal["date"].dt.isocalendar().week.astype(int)
test_with_seasonal = test_with_seasonal.drop(columns=["date"])

train_with_seasonal["month"] = train_with_seasonal["date"].dt.month
train_with_seasonal["dayofweek"] = train_with_seasonal["date"].dt.dayofweek
train_with_seasonal["quarter"] = train_with_seasonal["date"].dt.quarter
train_with_seasonal["day"] = train_with_seasonal["date"].dt.day
train_with_seasonal["weekofyear"] = train_with_seasonal["date"].dt.isocalendar().week.astype(int)
train_with_seasonal = train_with_seasonal.drop(columns=["date"])

In [ ]:
test_with_seasonal.to_csv("test_all_with_seasonal.csv", index=False)
train_with_seasonal.to_csv("train_all_with_seasonal.csv", index=False)

In [ ]:
train = pd.read_csv('test_all_with_seasonal.csv')
test = pd.read_csv('train_all_with_seasonal.csv')

In [ ]:
if 'cluster_id' in test.columns:
    le = LabelEncoder()
    test['cluster_id'] = le.fit_transform(test['cluster_id'])

In [ ]:
if 'cluster_id' in train.columns:
    le = LabelEncoder()
    train['cluster_id'] = le.fit_transform(train['cluster_id'])

In [ ]:
test.to_csv('test_all_with_seasonal_cluster.csv', index=False)
train.to_csv('train_all_with_seasonal_cluster.csv', index=False)

In [ ]:
train = pd.read_csv('train_all_with_seasonal_cluster.csv')
test = pd.read_csv('test_all_with_seasonal_cluster.csv')

In [ ]:
train.head()

In [ ]:
target_col = "electricity_consumption"

In [ ]:
param_dist = {
    'n_estimators': [100, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9, 11],
    'learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2],
    'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.25, 0.5, 1, 5],
    'min_child_weight': [1, 2, 3, 5, 7, 10],
    'reg_alpha': [0, 0.001, 0.01, 0.1, 1],     # L1 regularization
    'reg_lambda': [0.1, 0.5, 1.0, 2.0, 5.0]    # L2 regularization
}

In [ ]:
def train_and_eval_tuned(df_input, label):
    X = df_input.drop(columns=[target_col])
    y = df_input[target_col]
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    xgb = XGBRegressor(objective="reg:squarederror", random_state=42)

    # Use RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_dist,
        n_iter=1000,  
        scoring='neg_root_mean_squared_error',
        cv=3,
        verbose=2,
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train, y_train)
    best_model = random_search.best_estimator_

    y_pred = best_model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print(f"Tuned {label} RMSE: {rmse:.4f}")
    print(f"Best params: {random_search.best_params_}")

    return best_model

In [ ]:
train_cluster_date = train.drop(columns=["ID"])

In [ ]:
model = train_and_eval_tuned(train_cluster_date, "With seasonal and clusterid features")

In [ ]:
X_test = test.drop(columns=["ID"]) 

In [ ]:
y_test_pred = model.predict(X_test)

In [ ]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "electricity_consumption": y_test_pred
})
submission.to_csv("submission15_1000_verbose2.csv", index=False)